# 07 — SOFA horario y fenotipo Sepsis-3

Este notebook materializa (si es necesario) y valida los artefactos `30_score/sofa_hourly` y `40_labels/*` de una ejecución versionada bajo `data/derived/sofa/<run_id>/`. Solo presenta **resultados agregados**: nunca imprime identificadores ni filas a nivel de paciente. El shock séptico permanece pendiente de congelar su proxy operativo.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile

import pandas as pd
from IPython.display import SVG, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from mimic_sepsis.artifacts import ArtifactStore

ARTIFACT_NAME = 'sofa_hourly'
SOFA_RUNS_ROOT = PROJECT_ROOT / 'data' / 'derived' / 'sofa'
BUILDER = PROJECT_ROOT / 'scripts' / 'build_demo_sofa_incremental.py'
DATA_VERSION = '2.2'

def available_score_stores():
    """Devuelve ejecuciones completas ordenadas por fecha y ruta."""
    candidates = []
    for manifest_path in SOFA_RUNS_ROOT.glob('*/30_score/sofa_hourly.manifest.json'):
        store = ArtifactStore(manifest_path.parent)
        try:
            manifest = store.validate(ARTIFACT_NAME)
        except Exception as exc:
            print(f'Se omite artefacto inválido {manifest_path}: {exc}')
            continue
        candidates.append((manifest.created_at_utc, str(manifest_path.parent), store))
    return sorted(candidates, key=lambda item: (item[0], item[1]))


## Materialización incremental

La lógica clínica vive en `src/` y el script versionado. Si faltan artefactos, se construyen desde la raíz del repositorio.

In [ ]:
candidates = available_score_stores()
if not candidates:
    if not BUILDER.exists():
        raise FileNotFoundError(f'Falta {BUILDER}.')
    completed = subprocess.run(
        [sys.executable, str(BUILDER)], cwd=PROJECT_ROOT,
        text=True, capture_output=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(f'No se pudieron construir los artefactos:\n{completed.stderr}')
    candidates = available_score_stores()
if not candidates:
    raise RuntimeError('No existe ninguna ejecución SOFA válida.')
_, _, STORE = candidates[-1]
print(f'Usando {STORE.root.relative_to(PROJECT_ROOT)}')


## Infección sospechada y Sepsis-3 agregadas

La definición primaria usa administración EMAR confirmada y cultivo de sangre. El baseline es el mínimo SOFA horario en las 48 h anteriores a `t_si`; si no existe se presume cero y se marca. `t0` es el primer incremento ≥2 en la ventana ±24 h. Las tablas a nivel de episodio permanecen protegidas en memoria.

In [ ]:
LABEL_NAMES = ('suspected_infection_pairs', 'sepsis_episodes', 'sepsis_stays', 'septic_shock_stays')
label_store = ArtifactStore(STORE.root.parent / '40_labels')
try:
    for name in LABEL_NAMES:
        label_store.validate(name)
except Exception:
    completed = subprocess.run(
        [sys.executable, str(BUILDER), '--stage', 'label', '--resume'],
        cwd=PROJECT_ROOT, text=True, capture_output=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(f'No se pudieron construir las etiquetas:\n{completed.stderr}')
    candidates = available_score_stores()
    _, _, STORE = candidates[-1]
    label_store = ArtifactStore(STORE.root.parent / '40_labels')
    for name in LABEL_NAMES:
        label_store.validate(name)

pairs = label_store.read_dataframe('suspected_infection_pairs')
episodes = label_store.read_dataframe('sepsis_episodes')
sepsis_stays = label_store.read_dataframe('sepsis_stays')
shock_stays = label_store.read_dataframe('septic_shock_stays')


In [ ]:
phenotype_summary = pd.DataFrame({
    'metric': [
        'pares antibiótico–cultivo', 'ingresos con pares',
        'filas par–estancia evaluadas', 'pares sin UCI solapada',
        'episodios sin SOFA agudo', 'estancias con Sepsis-3',
        'episodios con baseline cero supuesto',
        'episodios con cobertura aguda completa', 'estancias con proxy de shock séptico',
    ],
    'count': [
        len(pairs), pairs['hadm_id'].nunique(), len(episodes),
        episodes['exclusion_reason'].eq('no_overlapping_icu_stay').sum(),
        episodes['exclusion_reason'].eq('no_acute_sofa_hours').sum(),
        len(sepsis_stays), episodes['baseline_assumed_zero'].sum(),
        episodes['acute_window_covered'].sum(), shock_stays['septic_shock'].sum(),
    ],
})
phenotype_summary


## Selección determinista del artefacto SOFA

Tras construir y validar las etiquetas, se vuelve a seleccionar la ejecución más reciente por fecha UTC del manifiesto y ruta, sin depender del estado histórico del kernel.

In [ ]:
candidates = available_score_stores()
if not candidates:
    if not BUILDER.exists():
        raise FileNotFoundError(
            f'Falta {BUILDER}. Restaura el script versionado antes de continuar.'
        )
    command = [sys.executable, str(BUILDER)]
    completed = subprocess.run(
        command, cwd=PROJECT_ROOT, text=True, capture_output=True
    )
    if completed.returncode != 0:
        raise RuntimeError(
            'No se pudo construir el SOFA incremental. Ejecuta desde la raíz:\n'
            f'  {sys.executable} scripts/build_demo_sofa_incremental.py\n\n'
            f'stdout:\n{completed.stdout}\n\nstderr:\n{completed.stderr}'
        )
    print(completed.stdout.strip() or 'Artefacto construido.')
    candidates = available_score_stores()
    if not candidates:
        raise RuntimeError('El constructor terminó pero no publicó 30_score/sofa_hourly.')

# Selección determinista: fecha UTC del manifiesto y ruta como desempate.
# Los run_id son hashes de configuración; normalmente existirá un único candidato.
_, _, STORE = candidates[-1]
print(f'Usando {STORE.root.relative_to(PROJECT_ROOT)} ({len(candidates)} ejecución/es válida/s)')


## Integridad y procedencia

Se comprueban checksum, esquema y número de filas antes de leer. La salida se limita a metadatos no identificables.

In [ ]:
manifest = STORE.validate(ARTIFACT_NAME)
assert manifest.data_version == DATA_VERSION, (
    f'Versión inesperada: {manifest.data_version!r}; se esperaba {DATA_VERSION!r}'
)
pd.DataFrame([{
    'artifact': manifest.artifact,
    'data_version': manifest.data_version,
    'code_version': manifest.code_version,
    'rows': manifest.rows,
    'n_columns': len(manifest.columns),
    'created_at_utc': manifest.created_at_utc,
    'sha256_prefix': manifest.sha256[:12],
}])


In [ ]:
# El marco contiene datos protegidos: se usa en memoria y nunca se muestra directamente.
sofa = STORE.read_dataframe(ARTIFACT_NAME)
required = {'sofa_total', 'sofa_complete', 'missing_components'}
missing = required.difference(sofa.columns)
if missing:
    raise ValueError(f'Esquema SOFA incompleto; faltan: {sorted(missing)}')

component_columns = [
    f'sofa_{name}' for name in
    ('respiratory', 'coagulation', 'liver', 'cardiovascular', 'cns', 'renal')
]
missing_components = set(component_columns).difference(sofa.columns)
if missing_components:
    raise ValueError(f'Faltan componentes: {sorted(missing_components)}')


## Completitud agregada

El denominador es el número de horas UCI del artefacto, no el número de pacientes. La completitud estricta exige los seis componentes; `sofa_total` conserva el convenio MIMIC de puntuar como cero los componentes ausentes.

In [ ]:
n_hours = len(sofa)
completeness = pd.DataFrame({
    'component': [column.removeprefix('sofa_') for column in component_columns],
    'hours_observed': [int(sofa[column].notna().sum()) for column in component_columns],
})
completeness['hours_total'] = n_hours
completeness['percent_observed'] = (
    100 * completeness['hours_observed'] / completeness['hours_total']
).round(1)
completeness


In [ ]:
missingness = (
    sofa['missing_components'].value_counts(dropna=False).sort_index()
    .rename_axis('missing_components').rename('hours').reset_index()
)
missingness['percent_hours'] = (100 * missingness['hours'] / n_hours).round(1)
missingness


## Distribución agregada del SOFA

La tabla contiene únicamente frecuencias por puntuación. Se usa como única fuente para ambas implementaciones gráficas.

In [ ]:
distribution = (
    sofa['sofa_total'].value_counts(dropna=False).sort_index()
    .rename_axis('sofa_total').rename('hours').reset_index()
)
distribution['percent_hours'] = (100 * distribution['hours'] / n_hours).round(2)
distribution


### Estilo Python (`matplotlib`)

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise ImportError(
        'Falta matplotlib. Actualiza el entorno con environment.yml o instala .[analysis].'
    ) from exc

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(distribution['sofa_total'].astype(str), distribution['percent_hours'], color='#2878B5')
ax.set(title='Distribución horaria del SOFA (convención MIMIC)', xlabel='SOFA total', ylabel='Horas UCI (%)')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()


### Estilo R (`ggplot2`)

Un notebook tiene un solo kernel principal. Para mantener Python como orquestador y usar el `ggplot2` real (no una emulación), la celda llama al `Rscript` del mismo entorno y muestra el SVG resultante.

In [ ]:
rscript = shutil.which('Rscript')
if rscript is None:
    raise RuntimeError('Rscript no está disponible; recrea el entorno desde environment.yml.')

r_code = r'''
args <- commandArgs(trailingOnly = TRUE)
suppressPackageStartupMessages(library(ggplot2))
d <- read.csv(args[[1]], check.names = FALSE)
d$sofa_total <- factor(d$sofa_total, levels = d$sofa_total)
p <- ggplot(d, aes(x = sofa_total, y = percent_hours)) +
  geom_col(fill = '#2878B5', width = 0.8) +
  labs(title = 'Distribución horaria del SOFA (convención MIMIC)',
       x = 'SOFA total', y = 'Horas UCI (%)') +
  theme_minimal(base_size = 12) +
  theme(panel.grid.minor = element_blank(), plot.title.position = 'plot')
ggsave(args[[2]], p, width = 9, height = 4.8, units = 'in', device = grDevices::svg)
'''
with tempfile.TemporaryDirectory() as directory:
    directory = Path(directory)
    aggregate_csv = directory / 'sofa_distribution_aggregate.csv'
    figure_svg = directory / 'sofa_distribution_ggplot2.svg'
    distribution.to_csv(aggregate_csv, index=False)
    completed = subprocess.run(
        [rscript, '-e', r_code, str(aggregate_csv), str(figure_svg)],
        text=True, capture_output=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(f'ggplot2 falló:\n{completed.stderr}')
    display(SVG(filename=str(figure_svg)))


## Criterio para avanzar

1. El manifiesto y checksum deben validar sin excepciones.
2. La completitud por componente y la distribución deben ser clínicamente plausibles y quedar revisadas.
3. Las diferencias entre gráficos son de estilo; números y denominadores deben coincidir porque comparten la misma tabla agregada.
4. Los pares y etiquetas Sepsis-3 deben validar y sus exclusiones/cobertura deben revisarse antes de escalar.
5. El siguiente paso congela el proxy de shock séptico y prepara el análisis descriptivo; los 519 pares del demo no equivalen a episodios independientes.